In [ ]:
%pip install python-docx

In [ ]:
import docx
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_ALIGN_VERTICAL
from docx.oxml import parse_xml, OxmlElement
from docx.oxml.ns import nsdecls, qn

def create_report():
    doc = Document()

    # ---------------------------------------------------------
    # PAGE SETUP: 1-inch margins
    # ---------------------------------------------------------
    for section in doc.sections:
        section.top_margin = Inches(1.0)
        section.bottom_margin = Inches(1.0)
        section.left_margin = Inches(1.0)
        section.right_margin = Inches(1.0)

    # ---------------------------------------------------------
    # GLOBAL STYLES & TYPOGRAPHY
    # ---------------------------------------------------------
    BLACK = RGBColor(0, 0, 0)

    # Normal Style
    normal_style = doc.styles['Normal']
    normal_style.font.name = 'Times New Roman'
    normal_style.font.size = Pt(12)
    normal_style.font.color.rgb = BLACK
    normal_style.paragraph_format.line_spacing = 1.15
    normal_style.paragraph_format.space_after = Pt(4)

    # Helper Functions
    def add_title_line(text, size=14, bold=True, align=WD_ALIGN_PARAGRAPH.CENTER, space_after=4):
        p = doc.add_paragraph()
        p.alignment = align
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(space_after)
        p.paragraph_format.line_spacing = 1.15
        run = p.add_run(text)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(size)
        run.font.bold = bold
        run.font.color.rgb = BLACK
        return p

    def add_h1(text):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(12)
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        p.paragraph_format.keep_with_next = True
        run = p.add_run(text)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(14)
        run.font.bold = True
        run.font.color.rgb = BLACK
        return p

    def add_h2(text):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(10)
        p.paragraph_format.space_after = Pt(3)
        p.paragraph_format.line_spacing = 1.15
        p.paragraph_format.keep_with_next = True
        run = p.add_run(text)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12)
        run.font.bold = True
        run.font.color.rgb = BLACK
        return p

    def add_body(text, bold_prefix=None, italic=False):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        if bold_prefix:
            run_b = p.add_run(bold_prefix)
            run_b.font.name = 'Times New Roman'
            run_b.font.size = Pt(12)
            run_b.font.bold = True
            run_b.font.color.rgb = BLACK
        run = p.add_run(text)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12)
        run.font.italic = italic
        run.font.color.rgb = BLACK
        return p

    def add_bullet(text, bold_prefix=None):
        p = doc.add_paragraph(style='List Bullet')
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(3)
        p.paragraph_format.line_spacing = 1.15
        if bold_prefix:
            run_b = p.add_run(bold_prefix)
            run_b.font.name = 'Times New Roman'
            run_b.font.size = Pt(12)
            run_b.font.bold = True
            run_b.font.color.rgb = BLACK
        run = p.add_run(text)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12)
        run.font.color.rgb = BLACK
        return p

    def add_code_block(code_text):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(4)
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.0
        run = p.add_run(code_text)
        run.font.name = 'Courier New'
        run.font.size = Pt(9.5)
        run.font.color.rgb = BLACK
        return p

    def set_cell_margins(cell, top=100, bottom=100, left=150, right=150):
        tcPr = cell._tc.get_or_add_tcPr()
        tcMar = OxmlElement('w:tcMar')
        for m, val in [('top', top), ('bottom', bottom), ('left', left), ('right', right)]:
            node = OxmlElement(f'w:{m}')
            node.set(qn('w:w'), str(val))
            node.set(qn('w:type'), 'dxa')
            tcMar.append(node)
        tcPr.append(tcMar)

    def set_table_borders(table):
        tblPr = table._tbl.tblPr
        tblBorders = parse_xml(
            r'<w:tblBorders %s>'
            r'  <w:top w:val="single" w:sz="6" w:space="0" w:color="000000"/>'
            r'  <w:left w:val="single" w:sz="6" w:space="0" w:color="000000"/>'
            r'  <w:bottom w:val="single" w:sz="6" w:space="0" w:color="000000"/>'
            r'  <w:right w:val="single" w:sz="6" w:space="0" w:color="000000"/>'
            r'  <w:insideH w:val="single" w:sz="4" w:space="0" w:color="000000"/>'
            r'  <w:insideV w:val="single" w:sz="4" w:space="0" w:color="000000"/>'
            r'</w:tblBorders>' % nsdecls('w')
        )
        tblPr.append(tblBorders)


    # ---------------------------------------------------------
    # TABLE OF CONTENTS
    # ---------------------------------------------------------
    add_title_line("TABLE OF CONTENTS", size=14, bold=True, align=WD_ALIGN_PARAGRAPH.CENTER, space_after=18)
    
    toc_entries = [
        ("1. Executive Summary & Project Background", "1"),
        ("   1.1 Institutional Context", "1"),
        ("   1.2 Problem Statement", "1"),
        ("   1.3 Project Objectives", "2"),
        ("2. Business Requirements & Key Performance Indicators (KPIs)", "3"),
        ("   2.1 Domain 1: Digital Asset Portfolio & Licensing ROI", "3"),
        ("   2.2 Domain 2: Smart Facility Utilization & Dynamic Operations", "3"),
        ("   2.3 Domain 3: Cross-College Equity & Resource Allocation", "4"),
        ("   2.4 Core Metric Summary Table", "4"),
        ("3. Star Schema Data Architecture & Dimensional Modeling", "5"),
        ("   3.1 Dimension Tables (Contextual Filters)", "5"),
        ("   3.2 Fact Tables (Transactional Process Measurements)", "6"),
        ("   3.3 Enterprise Dimensional Model Architecture", "6"),
        ("4. Database Implementation & SQL Analytical Engineering", "7"),
        ("   4.1 Synthetic Data Simulation Strategy", "7"),
        ("   4.2 Analytical View Engineering (Database-Side Transformation)", "8"),
        ("5. Business Intelligence & Power BI Layer", "11"),
        ("   5.1 Data Analysis Expressions (DAX) Measures", "11"),
        ("   5.2 Dashboard Architecture & Multi-Page Visual Layout", "12"),
        ("6. Analytical Findings & Strategic Business Insights", "14"),
        ("   6.1 Digital Subscriptions: Identifying Contractual Inefficiencies", "14"),
        ("   6.2 Physical Operations: Documenting Exam Crowding & Capacity Breach", "15"),
        ("   6.3 Cross-College Resource Disparities", "16"),
        ("7. Operational Recommendations & Strategic Action Plan", "17"),
        ("   7.1 Phase 1: Contract Restructuring & Capital Recovery (Months 1-3)", "17"),
        ("   7.2 Phase 2: Dynamic Facility Scheduling & Energy Optimization (Months 4-6)", "17"),
        ("   7.3 Phase 3: Capacity Rebalancing & Inter-College Equity (Months 7-12)", "18"),
        ("8. Technical Appendices", "19"),
        ("   Appendix A: Database Schema Definition (01_create_tables_schema.sql)", "19"),
        ("   Appendix B: Data Population & Views (02_populate_data_and_views.sql)", "21"),
        ("   Appendix C: Power BI DAX Measures Reference", "25")
    ]

    for title, page in toc_entries:
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(2)
        p.paragraph_format.line_spacing = 1.15
        run_t = p.add_run(title)
        run_t.font.name = 'Times New Roman'
        run_t.font.size = Pt(12)
        run_t.font.bold = True if title.startswith(('1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.')) else False
        run_t.font.color.rgb = BLACK
        
        # Leader dots calculation
        dots_count = max(5, 75 - len(title) - len(page))
        dots = " ." * (dots_count // 2)
        run_dots = p.add_run(f" {dots} ")
        run_dots.font.name = 'Times New Roman'
        run_dots.font.size = Pt(11)
        run_dots.font.color.rgb = BLACK

        run_p = p.add_run(page)
        run_p.font.name = 'Times New Roman'
        run_p.font.size = Pt(12)
        run_p.font.bold = True if title.startswith(('1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.')) else False
        run_p.font.color.rgb = BLACK

    doc.add_page_break()

    # ---------------------------------------------------------
    # SECTION 1
    # ---------------------------------------------------------
    add_h1("1. Executive Summary & Project Background")
    
    add_h2("1.1 Institutional Context")
    add_body(
        "The University of California, Berkeley library system operates as one of the preeminent public academic research "
        "library networks in the world. Encompassing major central hubs such as Doe Memorial Library and Moffitt Undergraduate "
        "Library alongside specialized repositories including the Bancroft Library, the Kresge Engineering Library, and the Long "
        "Business Library at Haas, the system supports tens of thousands of students, researchers, and faculty members. In recent "
        "fiscal cycles, campus leadership has confronted two competing operational pressures: escalating commercial digital "
        "subscription costs that consume significant portions of the operating budget, and severe foot-traffic volatility across "
        "branches during academic term surges."
    )

    add_h2("1.2 Problem Statement")
    add_body(
        "Despite generating millions of transactional data points—including physical turnstile security logs, circulation desk "
        "transactions, and digital proxy clickstreams—these data assets have historically operated within operational silos. "
        "Campus administration lacked a unified analytical platform capable of answering essential operational questions: Which "
        "commercial journal packages deliver justifiable cost-per-download value, and which represent contractual waste? What are "
        "the exact hourly building occupancy curves during exam periods? How equitably are physical spaces and digital collections "
        "utilized across Berkeley's independent colleges?"
    )

    add_h2("1.3 Project Objectives")
    add_body(
        "This project develops an enterprise data warehouse and business intelligence architecture designed to bridge the gap between "
        "low-level system transactions and senior leadership decision-making. The technical architecture delivers:"
    )
    add_bullet(" A standardized Star Schema Data Warehouse hosted in PostgreSQL that consolidates physical and digital library telemetry.")
    add_bullet(" A robust analytical transformation layer utilizing Common Table Expressions (CTEs) and SQL Window Functions to compute complex operational Key Performance Indicators.")
    add_bullet(" An executive three-page interactive Power BI dashboard providing provosts, facility managers, and collection directors with real-time operational visibility into digital contract ROI, building crowding, and cross-college resource allocation.")

    # ---------------------------------------------------------
    # SECTION 2
    # ---------------------------------------------------------
    add_h1("2. Business Requirements & Key Performance Indicators (KPIs)")
    add_body("To address institutional challenges, the analytical platform is structured across three core managerial domains:")

    add_h2("2.1 Domain 1: Digital Asset Portfolio & Licensing ROI")
    add_body(
        "Academic digital collections represent multi-million-dollar commitments. University leadership requires objective "
        "metrics to transition away from bundled commercial packages toward evidence-based contract management. Key indicators include:"
    )
    add_bullet("Cost-Per-Download (CPD): Measures the direct unit cost of access by dividing annual contract expenditure by verified research PDF downloads.", bold_prefix="• ")
    add_bullet("Portfolio Spend Allocation: Evaluates financial exposure categorized by publisher, subject domain, and licensing structure.", bold_prefix="• ")
    add_bullet("Strategic Contract Classification: Categorizes digital assets into operational tiers: Renew - High Value, Monitor - Moderate ROI, Renegotiate - High Cost/Use, and Cancel - Zero Usage.", bold_prefix="• ")

    add_h2("2.2 Domain 2: Smart Facility Utilization & Dynamic Operations")
    add_body(
        "Physical facilities incur heavy staffing, security, and climate-control (HVAC) expenses. Operational leads require "
        "granular time-series insights to optimize branch scheduling and maintain safe occupancy limits. Key indicators include:"
    )
    add_bullet("Hourly Net Flow: Evaluates the rate of headcount change by calculating entries minus exits within each hourly interval.", bold_prefix="• ")
    add_bullet("Cumulative Hourly Occupancy: Tracks the continuous, running headcount inside a branch across each 24-hour cycle.", bold_prefix="• ")
    add_bullet("Peak Capacity Utilization Percentage: Measures maximum observed occupancy against official fire-code seating limits.", bold_prefix="• ")

    add_h2("2.3 Domain 3: Cross-College Equity & Resource Allocation")
    add_body(
        "University funding is distributed across independent schools. Institutional planners require cross-departmental benchmarks "
        "to evaluate whether library resource consumption aligns equitably with academic enrollments across colleges and patron classifications."
    )

    add_h2("2.4 Core Metric Summary Table")
    add_body("Table 2.1 summarizes the core Key Performance Indicators implemented across the platform:")

    # Table 2.1
    t21_data = [
        ["KPI Name", "Formula / Logic", "Target Benchmark", "Institutional Purpose"],
        ["Gross Digital Spend", "SUM(annual_subscription_cost)", "Within Fiscal Budget", "Portfolio expenditure control"],
        ["Cost-Per-Download", "Annual Cost / Total PDF Downloads", "<= $15.00 per DL", "Contract valuation & audit"],
        ["Identified Savings", "SUM(Flagged Contract Costs)", "Maximize Capital Recovery", "Budget reallocation targeting"],
        ["Current Occupancy", "SUM(Entries - Exits) Running Total", "Dynamic by Branch", "Staffing and space scheduling"],
        ["Capacity Utilization", "(Current Occupancy / Capacity) * 100", "40% - 85% Safe Band", "Overcrowding & HVAC management"]
    ]
    t21 = doc.add_table(rows=len(t21_data), cols=4)
    set_table_borders(t21)
    t21.alignment = WD_TABLE_ALIGNMENT.CENTER
    for r_idx, row in enumerate(t21_data):
        for c_idx, val in enumerate(row):
            cell = t21.cell(r_idx, c_idx)
            set_cell_margins(cell, top=100, bottom=100, left=150, right=150)
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            run = p.add_run(val)
            run.font.name = 'Times New Roman'
            run.font.size = Pt(11)
            run.font.color.rgb = BLACK
            if r_idx == 0:
                run.font.bold = True
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    add_body("Table 2.1: Key Performance Indicators and Operational Benchmarks", italic=True)

    # ---------------------------------------------------------
    # SECTION 3
    # ---------------------------------------------------------
    add_h1("3. Star Schema Data Architecture & Dimensional Modeling")
    add_body(
        "To ensure analytical query performance and maintain dimensional integrity, the data warehouse was modeled following "
        "Ralph Kimball’s dimensional modeling methodology. The architecture isolates operational business processes into central "
        "Fact Tables connected to descriptive Dimension Tables via surrogate keys."
    )

    add_h2("3.1 Dimension Tables (Contextual Filters)")
    add_bullet("dim_branch: Catalogs physical facilities (Doe, Moffitt, Bancroft, Kresge, Haas), geographical zones, official capacities, and 24/7 indicators.", bold_prefix="• ")
    add_bullet("dim_digital_resource: Catalogs electronic databases, publisher details, subject domains, annual subscription costs, and licensing models.", bold_prefix="• ")
    add_bullet("dim_date_time: Establishes a uniform calendar grain across 720 hourly intervals across 30 days, capturing academic phases (Regular, Midterm Rush, Finals Week).", bold_prefix="• ")
    add_bullet("dim_patron: Stores anonymized patron profiles, classifications (Undergraduate, Graduate, Faculty), and college affiliations.", bold_prefix="• ")

    add_h2("3.2 Fact Tables (Transactional Process Measurements)")
    add_bullet("fact_physical_turnstile: Records individual gate events (ENTRY versus EXIT), timestamps, patron keys, branch keys, and gate numbers (120,000 records).", bold_prefix="• ")
    add_bullet("fact_physical_circulation: Captures book lending transactions, item categories (Course Reserve, General Stacks), checkout dates, and loan durations.", bold_prefix="• ")
    add_bullet("fact_digital_access: Logs network access interactions, research actions (SEARCH, ABSTRACT_VIEW, PDF_DOWNLOAD), and session durations.", bold_prefix="• ")

    add_h2("3.3 Enterprise Dimensional Model Architecture")
    add_body("Table 3.1 details the enterprise schema layout, primary keys, foreign keys, and operational table grains:")

    # Table 3.1
    t31_data = [
        ["Table Name", "Table Type", "Primary Key", "Foreign Keys", "Grain / Granularity"],
        ["dim_branch", "Dimension", "branch_id", "None", "One record per physical campus branch"],
        ["dim_digital_resource", "Dimension", "resource_id", "None", "One record per licensed electronic database"],
        ["dim_date_time", "Dimension", "date_key", "None", "One record per calendar date interval"],
        ["dim_patron", "Dimension", "patron_id", "None", "One record per anonymized institutional patron"],
        ["fact_physical_turnstile", "Fact", "turnstile_event_id", "patron_id, branch_id, date_key", "One record per turnstile swipe event"],
        ["fact_physical_circulation", "Fact", "circulation_id", "patron_id, branch_id, date_key", "One record per physical book checkout"],
        ["fact_digital_access", "Fact", "access_event_id", "patron_id, resource_id, date_key", "One record per proxy access session action"]
    ]
    t31 = doc.add_table(rows=len(t31_data), cols=5)
    set_table_borders(t31)
    t31.alignment = WD_TABLE_ALIGNMENT.CENTER
    for r_idx, row in enumerate(t31_data):
        for c_idx, val in enumerate(row):
            cell = t31.cell(r_idx, c_idx)
            set_cell_margins(cell, top=80, bottom=80, left=120, right=120)
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            run = p.add_run(val)
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            run.font.color.rgb = BLACK
            if r_idx == 0:
                run.font.bold = True
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    add_body("Table 3.1: Enterprise Star Schema Architecture and Relational Specifications", italic=True)

    # ---------------------------------------------------------
    # SECTION 4
    # ---------------------------------------------------------
    add_h1("4. Database Implementation & SQL Analytical Engineering")
    add_body(
        "The relational data warehouse was implemented on PostgreSQL and managed via pgAdmin 4. All analytical modeling was "
        "executed directly in the database engine using SQL Common Table Expressions (CTEs) and Window Functions."
    )

    add_h2("4.1 Synthetic Data Simulation Strategy")
    add_body(
        "To reflect realistic campus operations, a simulation script generated authentic transactions. Turnstile telemetry "
        "was calibrated to mirror university operating hours (7:30 AM to 10:30 PM), with weighted entry/exit probabilities reflecting "
        "morning arrivals (90% entry) and evening departures (78% to 92% exit). Event volumes were scaled across term phases: 20,000 "
        "events during Regular Term, 40,000 during Midterm Rush, and 60,000 during Finals Week (120,000 total rows)."
    )

    add_h2("4.2 Analytical View Engineering (Database-Side Transformation)")
    add_body(
        "To ensure optimal BI performance, three specialized views were compiled inside PostgreSQL to pre-aggregate complex window "
        "frames and calculations before loading into Power BI."
    )

    add_body("View 1: view_kpi_publisher_roi (Digital Contract Portfolio Valuation)", bold_prefix="SQL Implementation - ")
    add_code_block(
        "CREATE OR REPLACE VIEW view_kpi_publisher_roi AS\n"
        "WITH ResourceMetrics AS (\n"
        "    SELECT \n"
        "        r.resource_id, r.resource_title, r.publisher, r.subject_domain, \n"
        "        r.annual_subscription_cost, r.license_model,\n"
        "        COUNT(f.access_event_id) AS total_access_events,\n"
        "        COUNT(CASE WHEN f.access_action = 'PDF_DOWNLOAD' THEN 1 END) AS total_pdf_downloads,\n"
        "        COUNT(DISTINCT f.patron_id) AS unique_active_users\n"
        "    FROM dim_digital_resource r\n"
        "    LEFT JOIN fact_digital_access f ON r.resource_id = f.resource_id\n"
        "    GROUP BY r.resource_id, r.resource_title, r.publisher, r.subject_domain, r.annual_subscription_cost, r.license_model\n"
        ")\n"
        "SELECT \n"
        "    resource_id, resource_title, publisher, subject_domain, annual_subscription_cost, license_model,\n"
        "    total_access_events, total_pdf_downloads, unique_active_users,\n"
        "    CASE \n"
        "        WHEN total_pdf_downloads > 0 THEN ROUND(annual_subscription_cost / total_pdf_downloads, 2)\n"
        "        ELSE annual_subscription_cost \n"
        "    END AS cost_per_download_usd,\n"
        "    CASE \n"
        "        WHEN total_pdf_downloads = 0 THEN 'CANCEL - Zero Usage'\n"
        "        WHEN (annual_subscription_cost / NULLIF(total_pdf_downloads, 0)) > 80.00 THEN 'RENEGOTIATE - High Cost/Use'\n"
        "        WHEN (annual_subscription_cost / NULLIF(total_pdf_downloads, 0)) BETWEEN 15.00 AND 80.00 THEN 'MONITOR - Moderate ROI'\n"
        "        ELSE 'RENEW - High Value Portfolio'\n"
        "    END AS strategic_contract_action\n"
        "FROM ResourceMetrics;"
    )

    add_body("View 2: view_kpi_hourly_occupancy (Dynamic Cumulative Headcount via Window Functions)", bold_prefix="SQL Implementation - ")
    add_code_block(
        "CREATE OR REPLACE VIEW view_kpi_hourly_occupancy AS\n"
        "WITH HourlyFlow AS (\n"
        "    SELECT \n"
        "        t.branch_id, b.branch_name, b.total_capacity,\n"
        "        t.event_timestamp::DATE AS full_date,\n"
        "        EXTRACT(HOUR FROM t.event_timestamp)::INT AS hour_of_day,\n"
        "        d.academic_term_phase,\n"
        "        SUM(CASE WHEN t.event_type = 'ENTRY' THEN 1 ELSE 0 END) AS total_entries,\n"
        "        SUM(CASE WHEN t.event_type = 'EXIT' THEN 1 ELSE 0 END) AS total_exits\n"
        "    FROM fact_physical_turnstile t\n"
        "    JOIN dim_branch b ON t.branch_id = b.branch_id\n"
        "    JOIN dim_date_time d ON t.date_key = d.date_key\n"
        "    GROUP BY t.branch_id, b.branch_name, b.total_capacity, t.event_timestamp::DATE, EXTRACT(HOUR FROM t.event_timestamp)::INT, d.academic_term_phase\n"
        ")\n"
        "SELECT \n"
        "    branch_id, branch_name, total_capacity, full_date, hour_of_day, academic_term_phase,\n"
        "    total_entries, total_exits, (total_entries - total_exits) AS net_flow,\n"
        "    SUM(total_entries - total_exits) OVER (\n"
        "        PARTITION BY branch_id, full_date ORDER BY hour_of_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n"
        "    ) AS estimated_current_occupancy,\n"
        "    ROUND(\n"
        "        (GREATEST(0, SUM(total_entries - total_exits) OVER (\n"
        "            PARTITION BY branch_id, full_date ORDER BY hour_of_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n"
        "        ))::NUMERIC / total_capacity) * 100, 2\n"
        "    ) AS capacity_utilization_pct\n"
        "FROM HourlyFlow;"
    )

    add_body("View 3: view_kpi_college_usage_share (Cross-College Benchmarking Mart)", bold_prefix="SQL Implementation - ")
    add_code_block(
        "CREATE OR REPLACE VIEW view_kpi_college_usage_share AS\n"
        "SELECT \n"
        "    p.college_affiliation, p.patron_type,\n"
        "    COUNT(DISTINCT p.patron_id) AS total_registered_patrons,\n"
        "    COUNT(DISTINCT t.turnstile_event_id) AS total_library_visits,\n"
        "    COUNT(DISTINCT c.circulation_id) AS total_physical_books_borrowed,\n"
        "    COUNT(DISTINCT d.access_event_id) AS total_digital_sessions,\n"
        "    COUNT(CASE WHEN d.access_action = 'PDF_DOWNLOAD' THEN 1 END) AS total_research_downloads\n"
        "FROM dim_patron p\n"
        "LEFT JOIN fact_physical_turnstile t ON p.patron_id = t.patron_id\n"
        "LEFT JOIN fact_physical_circulation c ON p.patron_id = c.patron_id\n"
        "LEFT JOIN fact_digital_access d ON p.patron_id = d.patron_id\n"
        "GROUP BY p.college_affiliation, p.patron_type;"
    )

    # ---------------------------------------------------------
    # SECTION 5
    # ---------------------------------------------------------
    add_h1("5. Business Intelligence & Power BI Layer")
    add_body(
        "Power BI Desktop connects to the PostgreSQL data warehouse via native database import connectors. By loading pre-aggregated "
        "SQL views, the BI layer maintains maximum interactive responsiveness without executing complex multi-table joins in memory."
    )

    add_h2("5.1 Data Analysis Expressions (DAX) Measures")
    add_body("Key business metrics were authored using DAX to drive dynamic card visualizations and gauge dials:")
    add_code_block(
        "// 1. Total University Spend on Subscriptions\n"
        "Total Digital Spend = SUM(view_kpi_publisher_roi[annual_subscription_cost])\n\n"
        "// 2. Total Research PDF Downloads\n"
        "Total Downloads = SUM(view_kpi_publisher_roi[total_pdf_downloads])\n\n"
        "// 3. Overall Portfolio Cost-Per-Download\n"
        "Overall CPD = DIVIDE([Total Digital Spend], [Total Downloads], 0)\n\n"
        "// 4. Actionable Capital Recovery Target\n"
        "Potential Annual Savings = \n"
        "CALCULATE(\n"
        "    SUM(view_kpi_publisher_roi[annual_subscription_cost]),\n"
        "    view_kpi_publisher_roi[strategic_contract_action] IN {\"CANCEL - Zero Usage\", \"RENEGOTIATE - High Cost/Use\"}\n"
        ")\n\n"
        "// 5. Normalized Gauge Capacity Dial Metric\n"
        "Peak Capacity % = DIVIDE(MAX(view_kpi_hourly_occupancy[capacity_utilization_pct]), 100, 0)"
    )

    add_h2("5.2 Dashboard Architecture & Multi-Page Visual Layout")
    add_bullet("Page 1: Executive Suite (Provost & Deans): Features four primary KPI banner cards (Total Digital Spend: $589.00K, Total Downloads: 6,007, Overall CPD: $98.05, Potential Savings: $452.00K), a clustered horizontal bar chart benchmarking college usage share, and a donut chart displaying spend by licensing model.")
    add_bullet("Page 2: Digital ROI & Publisher Negotiations: Houses an interactive publisher slicer, a scatter plot mapping download volume against subscription costs (bubble size scaled by CPD), and a detailed strategic contract table with conditional color-coding.")
    add_bullet("Page 3: Smart Facilities & Space Optimization: Features branch and academic term slicers, a 24-hour occupancy line chart tracking daily curves, a capacity utilization gauge with an 85% warning threshold (surging to 115.60% during Finals Week), and an hourly entry heatmap matrix.")

    # ---------------------------------------------------------
    # SECTION 6
    # ---------------------------------------------------------
    add_h1("6. Analytical Findings & Strategic Business Insights")
    
    add_h2("6.1 Digital Subscriptions: Identifying Contractual Inefficiencies")
    add_body(
        "Analysis of the $589,000.00 annual subscription portfolio reveals extreme valuation disparities across vendors. "
        "The Historical Legal Treatises Micro-Archive represents critical contractual waste, costing $34,000.00 while generating "
        "only 21 verified downloads (CPD: $1,619.05). Similarly, BioOne Complete delivered 112 downloads against a $28,000.00 fee "
        "(CPD: $250.00). Conversely, core packages such as ScienceDirect ($185K cost, 1,060 downloads, CPD: $174.53) and IEEE Xplore "
        "($95K cost, 752 downloads, CPD: $126.33) demonstrate robust academic utility. In total, the system identified $452,000.00 in "
        "contracts flagged for cancellation or aggressive renegotiation."
    )

    add_h2("6.2 Physical Operations: Documenting Exam Crowding & Capacity Breach")
    add_body(
        "Turnstile telemetry captures clear diurnal flow patterns across operating hours (7:30 AM to 10:30 PM), characterized by "
        "heavy morning arrivals (7:30 AM - 11:00 AM), a steady peak plateau (12:00 PM - 4:00 PM), and gradual departures after 5:00 PM. "
        "During Regular Term schedules, peak capacity remains within safe boundaries (30% - 45%). However, during Finals Week, peak "
        "utilization breaches safe limits, reaching 115.60% at central hubs like Moffitt Library. This indicates severe seat shortages, "
        "whereas specialized repositories like Bancroft operate below 20% capacity throughout the term."
    )

    add_h2("6.3 Cross-College Resource Disparities")
    add_body(
        "Benchmarking demonstrates that the College of Engineering and the College of Letters & Science account for over 68% of physical "
        "gate entries and 70% of research article downloads. In contrast, the Haas School of Business exhibits concentrated digital "
        "usage of specialized business terminals (WRDS) with lower physical turnstile volume at main campus libraries, reflecting "
        "the decentralized autonomy of satellite facilities."
    )

    # ---------------------------------------------------------
    # SECTION 7
    # ---------------------------------------------------------
    add_h1("7. Operational Recommendations & Strategic Action Plan")
    add_body("To capitalize on analytical findings, campus leadership should execute a structured three-phase roadmap:")

    add_h2("7.1 Phase 1: Contract Restructuring & Capital Recovery (Months 1-3)")
    add_bullet("Cancel Niche Contracts: Terminate the Historical Legal Treatises Micro-Archive, recovering $34,000.00 immediately while routing specialized research needs through Inter-Library Loans (ILL).", bold_prefix="• ")
    add_bullet("Unbundle Commercial Packages: Present empirical cost-per-download data during publisher renewals with Elsevier and Springer Nature to break up high-cost bundles, targeting 20% to 30% savings across $418,000.00 in flagged subscriptions.", bold_prefix="• ")

    add_h2("7.2 Phase 2: Dynamic Facility Scheduling & Energy Optimization (Months 4-6)")
    add_bullet("Shift Staffing Schedules: Align front-desk and security staffing to match the 11:00 AM - 4:00 PM peak arrival window revealed in the entry heatmap matrix.", bold_prefix="• ")
    add_bullet("Smart HVAC Setbacks: Link turnstile sensor data with campus building automation systems to reduce heating and cooling output during low-occupancy evening and weekend hours, lowering utility expenses.", bold_prefix="• ")

    add_h2("7.3 Phase 3: Capacity Rebalancing & Inter-College Equity (Months 7-12)")
    add_bullet("Exam Overflow Management: Mitigate the 115.60% finals week capacity breach by opening temporary overflow study halls in underutilized academic buildings and displaying live seat availability on campus apps.", bold_prefix="• ")
    add_bullet("Proportional Cost Sharing: Utilize college usage benchmarks to establish shared digital funding models, ensuring departmental contributions correspond to verified resource consumption.", bold_prefix="• ")

    add_body("Table 7.1 details the strategic operational roadmap, milestones, and expected outcomes:")

    # Table 7.1
    t71_data = [
        ["Implementation Phase", "Target Timeline", "Core Initiatives & Deliverables", "Expected Institutional Impact"],
        ["Phase 1: Contract Restructuring", "Months 1 - 3", "Cancel zero-usage legal archive; unbundle commercial journal packages using CPD data.", "Direct budget recovery of $34K to $125K annually."],
        ["Phase 2: Facilities Optimization", "Months 4 - 6", "Realign desk staffing to 11 AM - 4 PM; integrate turnstile counts with campus HVAC.", "Reduced energy overhead and optimized student desk coverage."],
        ["Phase 3: Space Rebalancing", "Months 7 - 12", "Open finals overflow study spaces; implement cross-college subscription cost sharing.", "Elimination of seat crowding and equitable departmental budgeting."]
    ]
    t71 = doc.add_table(rows=len(t71_data), cols=4)
    set_table_borders(t71)
    t71.alignment = WD_TABLE_ALIGNMENT.CENTER
    for r_idx, row in enumerate(t71_data):
        for c_idx, val in enumerate(row):
            cell = t71.cell(r_idx, c_idx)
            set_cell_margins(cell, top=100, bottom=100, left=150, right=150)
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            run = p.add_run(val)
            run.font.name = 'Times New Roman'
            run.font.size = Pt(11)
            run.font.color.rgb = BLACK
            if r_idx == 0:
                run.font.bold = True
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    add_body("Table 7.1: Strategic Implementation Roadmap and Milestone Targets", italic=True)

    # ---------------------------------------------------------
    # SECTION 8
    # ---------------------------------------------------------
    add_h1("8. Technical Appendices")
    
    add_h2("Appendix A: Database Schema Definition (01_create_tables_schema.sql)")
    add_body("Complete DDL script establishing the star schema tables, relationships, and constraints in PostgreSQL:")
    add_code_block(
        "DROP TABLE IF EXISTS fact_digital_access CASCADE;\n"
        "DROP TABLE IF EXISTS fact_physical_circulation CASCADE;\n"
        "DROP TABLE IF EXISTS fact_physical_turnstile CASCADE;\n"
        "DROP TABLE IF EXISTS dim_patron CASCADE;\n"
        "DROP TABLE IF EXISTS dim_date_time CASCADE;\n"
        "DROP TABLE IF EXISTS dim_digital_resource CASCADE;\n"
        "DROP TABLE IF EXISTS dim_branch CASCADE;\n\n"
        "CREATE TABLE dim_branch (\n"
        "    branch_id VARCHAR(16) PRIMARY KEY,\n"
        "    branch_name VARCHAR(128) NOT NULL,\n"
        "    campus_zone VARCHAR(64) NOT NULL,\n"
        "    total_capacity INT NOT NULL CHECK (total_capacity > 0),\n"
        "    is_24_7_service BOOLEAN NOT NULL DEFAULT FALSE\n"
        ");\n\n"
        "CREATE TABLE dim_digital_resource (\n"
        "    resource_id VARCHAR(32) PRIMARY KEY,\n"
        "    resource_title VARCHAR(256) NOT NULL,\n"
        "    publisher VARCHAR(128) NOT NULL,\n"
        "    subject_domain VARCHAR(64) NOT NULL,\n"
        "    annual_subscription_cost NUMERIC(12, 2) NOT NULL CHECK (annual_subscription_cost >= 0),\n"
        "    license_model VARCHAR(64) NOT NULL\n"
        ");\n\n"
        "CREATE TABLE dim_date_time (\n"
        "    date_key INT PRIMARY KEY,\n"
        "    full_date DATE NOT NULL,\n"
        "    hour_of_day INT NOT NULL CHECK (hour_of_day BETWEEN 0 AND 23),\n"
        "    day_of_week VARCHAR(16) NOT NULL,\n"
        "    is_weekend BOOLEAN NOT NULL,\n"
        "    academic_term_phase VARCHAR(32) NOT NULL\n"
        ");\n\n"
        "CREATE TABLE dim_patron (\n"
        "    patron_id VARCHAR(32) PRIMARY KEY,\n"
        "    patron_type VARCHAR(32) NOT NULL,\n"
        "    college_affiliation VARCHAR(128) NOT NULL,\n"
        "    academic_standing VARCHAR(32) NOT NULL\n"
        ");\n\n"
        "CREATE TABLE fact_physical_turnstile (\n"
        "    turnstile_event_id BIGSERIAL PRIMARY KEY,\n"
        "    patron_id VARCHAR(32) NOT NULL REFERENCES dim_patron(patron_id),\n"
        "    branch_id VARCHAR(16) NOT NULL REFERENCES dim_branch(branch_id),\n"
        "    date_key INT NOT NULL REFERENCES dim_date_time(date_key),\n"
        "    event_timestamp TIMESTAMP NOT NULL,\n"
        "    event_type VARCHAR(8) NOT NULL CHECK (event_type IN ('ENTRY', 'EXIT')),\n"
        "    turnstile_gate_number INT NOT NULL\n"
        ");\n\n"
        "CREATE TABLE fact_physical_circulation (\n"
        "    circulation_id BIGSERIAL PRIMARY KEY,\n"
        "    patron_id VARCHAR(32) NOT NULL REFERENCES dim_patron(patron_id),\n"
        "    branch_id VARCHAR(16) NOT NULL REFERENCES dim_branch(branch_id),\n"
        "    date_key INT NOT NULL REFERENCES dim_date_time(date_key),\n"
        "    checkout_timestamp TIMESTAMP NOT NULL,\n"
        "    return_timestamp TIMESTAMP,\n"
        "    item_category VARCHAR(64) NOT NULL,\n"
        "    loan_duration_days INT NOT NULL CHECK (loan_duration_days >= 0)\n"
        ");\n\n"
        "CREATE TABLE fact_digital_access (\n"
        "    access_event_id BIGSERIAL PRIMARY KEY,\n"
        "    patron_id VARCHAR(32) NOT NULL REFERENCES dim_patron(patron_id),\n"
        "    resource_id VARCHAR(32) NOT NULL REFERENCES dim_digital_resource(resource_id),\n"
        "    date_key INT NOT NULL REFERENCES dim_date_time(date_key),\n"
        "    access_timestamp TIMESTAMP NOT NULL,\n"
        "    access_action VARCHAR(32) NOT NULL CHECK (access_action IN ('SEARCH', 'ABSTRACT_VIEW', 'PDF_DOWNLOAD')),\n"
        "    session_duration_sec INT NOT NULL CHECK (session_duration_sec >= 0),\n"
        "    access_network VARCHAR(64) NOT NULL\n"
        ");\n\n"
        "CREATE INDEX idx_turnstile_branch_date ON fact_physical_turnstile(branch_id, date_key);\n"
        "CREATE INDEX idx_turnstile_timestamp ON fact_physical_turnstile(event_timestamp);\n"
        "CREATE INDEX idx_digital_resource_action ON fact_digital_access(resource_id, access_action);"
    )

    add_h2("Appendix B: Data Population & Views (02_populate_data_and_views.sql)")
    add_body("Complete DML and View script populating synthetic telemetry and compiling analytical views:")
    add_code_block(
        "INSERT INTO dim_branch (branch_id, branch_name, campus_zone, total_capacity, is_24_7_service) VALUES\n"
        "('BR_DOE', 'Doe Memorial Library', 'Central Campus', 1200, FALSE),\n"
        "('BR_MOFF', 'Moffitt Undergraduate Library', 'Central Campus', 1800, TRUE),\n"
        "('BR_BANC', 'Bancroft Special Collections', 'Central Campus', 250, FALSE),\n"
        "('BR_ENGR', 'Kresge Engineering Library', 'North Side', 600, FALSE),\n"
        "('BR_HAAS', 'Long Business Library (Haas)', 'South Side', 450, FALSE)\n"
        "ON CONFLICT (branch_id) DO NOTHING;\n\n"
        "INSERT INTO dim_digital_resource (resource_id, resource_title, publisher, subject_domain, annual_subscription_cost, license_model) VALUES\n"
        "('RES_SD_CORE', 'ScienceDirect Complete College Package', 'Elsevier', 'Multidisciplinary', 185000.00, 'Big Deal Bundle'),\n"
        "('RES_IEEE_XPL', 'IEEE Xplore Digital Library', 'IEEE', 'Engineering', 95000.00, 'Core Direct'),\n"
        "('RES_JSTOR_ARC', 'JSTOR Arts & Sciences Collection', 'ITHAKA / JSTOR', 'Humanities', 42000.00, 'Core Direct'),\n"
        "('RES_SPR_NAT', 'SpringerLink Scientific Journals', 'Springer Nature', 'Sciences', 130000.00, 'Big Deal Bundle'),\n"
        "('RES_WRDS_BUS', 'Wharton Research Data Services (WRDS)', 'Wharton / Multiple', 'Business', 75000.00, 'Core Direct'),\n"
        "('RES_BIO_ONE', 'BioOne Complete Research Package', 'BioOne', 'Life Sciences', 28000.00, 'Core Direct'),\n"
        "('RES_OLD_LEG', 'Historical Legal Treatises Micro-Archive', 'Gale / Cengage', 'Law', 34000.00, 'Big Deal Bundle')\n"
        "ON CONFLICT (resource_id) DO NOTHING;\n\n"
        "INSERT INTO dim_date_time (date_key, full_date, hour_of_day, day_of_week, is_weekend, academic_term_phase)\n"
        "SELECT \n"
        "    to_char(d, 'YYYYMMDD')::INT AS date_key, d::DATE AS full_date, 0 AS hour_of_day,\n"
        "    TRIM(to_char(d, 'Day')) AS day_of_week, EXTRACT(ISODOW FROM d) IN (6, 7) AS is_weekend,\n"
        "    CASE \n"
        "        WHEN d < '2026-09-10' THEN 'Regular Term'\n"
        "        WHEN d BETWEEN '2026-09-10' AND '2026-09-20' THEN 'Midterm Rush'\n"
        "        ELSE 'Finals Week'\n"
        "    END AS academic_term_phase\n"
        "FROM generate_series('2026-09-01'::DATE, '2026-09-30'::DATE, '1 day'::interval) d\n"
        "ON CONFLICT (date_key) DO NOTHING;\n\n"
        "INSERT INTO dim_patron (patron_id, patron_type, college_affiliation, academic_standing)\n"
        "SELECT \n"
        "    'PATRON_' || (10000 + i) AS patron_id,\n"
        "    (ARRAY['Undergraduate', 'Undergraduate', 'Undergraduate', 'Graduate', 'Graduate', 'Faculty', 'Postdoctoral'])[floor(random()*7 + 1)],\n"
        "    (ARRAY['Haas School of Business', 'College of Engineering', 'Letters & Science', 'School of Law', 'Computing & Data Science'])[floor(random()*5 + 1)],\n"
        "    (ARRAY['Freshman', 'Sophomore', 'Junior', 'Senior', 'Masters', 'Doctoral'])[floor(random()*6 + 1)]\n"
        "FROM generate_series(1, 500) i\n"
        "ON CONFLICT (patron_id) DO NOTHING;\n\n"
        "TRUNCATE TABLE fact_physical_turnstile;\n\n"
        "INSERT INTO fact_physical_turnstile (patron_id, branch_id, date_key, event_timestamp, event_type, turnstile_gate_number)\n"
        "WITH SurgeEvents AS (\n"
        "    SELECT \n"
        "        (ARRAY['BR_MOFF', 'BR_MOFF', 'BR_DOE', 'BR_DOE', 'BR_ENGR', 'BR_HAAS', 'BR_BANC'])[floor(random()*7 + 1)] AS branch_id,\n"
        "        '2026-09-01 07:30:00'::timestamp + (floor(random() * 9) * interval '1 day') + (floor(random() * 15) * interval '1 hour') + (floor(random() * 60) * interval '1 minute') AS ts\n"
        "    FROM generate_series(1, 20000)\n"
        "    UNION ALL\n"
        "    SELECT \n"
        "        (ARRAY['BR_MOFF', 'BR_MOFF', 'BR_MOFF', 'BR_DOE', 'BR_DOE', 'BR_ENGR', 'BR_HAAS', 'BR_BANC'])[floor(random()*8 + 1)] AS branch_id,\n"
        "        '2026-09-10 07:30:00'::timestamp + (floor(random() * 10) * interval '1 day') + (floor(random() * 15) * interval '1 hour') + (floor(random() * 60) * interval '1 minute') AS ts\n"
        "    FROM generate_series(1, 40000)\n"
        "    UNION ALL\n"
        "    SELECT \n"
        "        (ARRAY['BR_MOFF', 'BR_MOFF', 'BR_MOFF', 'BR_DOE', 'BR_DOE', 'BR_ENGR', 'BR_HAAS', 'BR_BANC'])[floor(random()*8 + 1)] AS branch_id,\n"
        "        '2026-09-20 07:30:00'::timestamp + (floor(random() * 10) * interval '1 day') + (floor(random() * 15) * interval '1 hour') + (floor(random() * 60) * interval '1 minute') AS ts\n"
        "    FROM generate_series(1, 60000)\n"
        ")\n"
        "SELECT \n"
        "    'PATRON_' || (10000 + floor(random()*500 + 1)) AS patron_id,\n"
        "    s.branch_id, to_char(s.ts, 'YYYYMMDD')::INT AS date_key, s.ts AS event_timestamp,\n"
        "    CASE \n"
        "        WHEN EXTRACT(HOUR FROM s.ts) BETWEEN 7 AND 12 THEN CASE WHEN random() < 0.90 THEN 'ENTRY' ELSE 'EXIT' END\n"
        "        WHEN EXTRACT(HOUR FROM s.ts) BETWEEN 13 AND 16 THEN CASE WHEN random() < 0.52 THEN 'ENTRY' ELSE 'EXIT' END\n"
        "        WHEN EXTRACT(HOUR FROM s.ts) BETWEEN 17 AND 20 THEN CASE WHEN random() < 0.22 THEN 'ENTRY' ELSE 'EXIT' END\n"
        "        ELSE CASE WHEN random() < 0.08 THEN 'ENTRY' ELSE 'EXIT' END\n"
        "    END AS event_type,\n"
        "    floor(random()*4 + 1)::INT AS turnstile_gate_number\n"
        "FROM SurgeEvents s;\n\n"
        "INSERT INTO fact_physical_circulation (patron_id, branch_id, date_key, checkout_timestamp, return_timestamp, item_category, loan_duration_days)\n"
        "SELECT \n"
        "    'PATRON_' || (10000 + floor(random()*500 + 1)) AS patron_id,\n"
        "    (ARRAY['BR_DOE', 'BR_MOFF', 'BR_ENGR', 'BR_HAAS', 'BR_BANC'])[floor(random()*5 + 1)],\n"
        "    to_char(ts, 'YYYYMMDD')::INT, ts, ts + (floor(random()*14 + 1) * interval '1 day'),\n"
        "    (ARRAY['Course Reserve', 'General Stacks', 'General Stacks', 'AudioVisual'])[floor(random()*4 + 1)],\n"
        "    floor(random()*14 + 1)::INT\n"
        "FROM (SELECT '2026-09-01 09:00:00'::timestamp + (random() * 29 * 86400 * interval '1 second') AS ts FROM generate_series(1, 1000)) sub;\n\n"
        "INSERT INTO fact_digital_access (patron_id, resource_id, date_key, access_timestamp, access_action, session_duration_sec, access_network)\n"
        "SELECT \n"
        "    'PATRON_' || (10000 + floor(random()*500 + 1)) AS patron_id,\n"
        "    CASE \n"
        "        WHEN r <= 0.35 THEN 'RES_SD_CORE'\n"
        "        WHEN r <= 0.60 THEN 'RES_IEEE_XPL'\n"
        "        WHEN r <= 0.75 THEN 'RES_JSTOR_ARC'\n"
        "        WHEN r <= 0.87 THEN 'RES_SPR_NAT'\n"
        "        WHEN r <= 0.95 THEN 'RES_WRDS_BUS'\n"
        "        WHEN r <= 0.99 THEN 'RES_BIO_ONE'\n"
        "        ELSE 'RES_OLD_LEG'\n"
        "    END,\n"
        "    to_char(ts, 'YYYYMMDD')::INT, ts,\n"
        "    (ARRAY['SEARCH', 'ABSTRACT_VIEW', 'PDF_DOWNLOAD', 'PDF_DOWNLOAD'])[floor(random()*4 + 1)],\n"
        "    floor(random()*1500 + 30)::INT, (ARRAY['AirBears_WiFi', 'OffCampus_EZproxy'])[floor(random()*2 + 1)]\n"
        "FROM (SELECT random() AS r, '2026-09-01 00:00:00'::timestamp + (random() * 29 * 86400 * interval '1 second') AS ts FROM generate_series(1, 6000)) sub;\n\n"
        "CREATE OR REPLACE VIEW view_kpi_publisher_roi AS\n"
        "WITH ResourceMetrics AS (\n"
        "    SELECT \n"
        "        r.resource_id, r.resource_title, r.publisher, r.subject_domain, r.annual_subscription_cost, r.license_model,\n"
        "        COUNT(f.access_event_id) AS total_access_events,\n"
        "        COUNT(CASE WHEN f.access_action = 'PDF_DOWNLOAD' THEN 1 END) AS total_pdf_downloads,\n"
        "        COUNT(DISTINCT f.patron_id) AS unique_active_users\n"
        "    FROM dim_digital_resource r\n"
        "    LEFT JOIN fact_digital_access f ON r.resource_id = f.resource_id\n"
        "    GROUP BY r.resource_id, r.resource_title, r.publisher, r.subject_domain, r.annual_subscription_cost, r.license_model\n"
        ")\n"
        "SELECT \n"
        "    resource_id, resource_title, publisher, subject_domain, annual_subscription_cost, license_model,\n"
        "    total_access_events, total_pdf_downloads, unique_active_users,\n"
        "    CASE WHEN total_pdf_downloads > 0 THEN ROUND(annual_subscription_cost / total_pdf_downloads, 2) ELSE annual_subscription_cost END AS cost_per_download_usd,\n"
        "    CASE \n"
        "        WHEN total_pdf_downloads = 0 THEN 'CANCEL - Zero Usage'\n"
        "        WHEN (annual_subscription_cost / NULLIF(total_pdf_downloads, 0)) > 80.00 THEN 'RENEGOTIATE - High Cost/Use'\n"
        "        WHEN (annual_subscription_cost / NULLIF(total_pdf_downloads, 0)) BETWEEN 15.00 AND 80.00 THEN 'MONITOR - Moderate ROI'\n"
        "        ELSE 'RENEW - High Value Portfolio'\n"
        "    END AS strategic_contract_action\n"
        "FROM ResourceMetrics;\n\n"
        "CREATE OR REPLACE VIEW view_kpi_hourly_occupancy AS\n"
        "WITH HourlyFlow AS (\n"
        "    SELECT \n"
        "        t.branch_id, b.branch_name, b.total_capacity, t.event_timestamp::DATE AS full_date,\n"
        "        EXTRACT(HOUR FROM t.event_timestamp)::INT AS hour_of_day, d.academic_term_phase,\n"
        "        SUM(CASE WHEN t.event_type = 'ENTRY' THEN 1 ELSE 0 END) AS total_entries,\n"
        "        SUM(CASE WHEN t.event_type = 'EXIT' THEN 1 ELSE 0 END) AS total_exits\n"
        "    FROM fact_physical_turnstile t\n"
        "    JOIN dim_branch b ON t.branch_id = b.branch_id\n"
        "    JOIN dim_date_time d ON t.date_key = d.date_key\n"
        "    GROUP BY t.branch_id, b.branch_name, b.total_capacity, t.event_timestamp::DATE, EXTRACT(HOUR FROM t.event_timestamp)::INT, d.academic_term_phase\n"
        ")\n"
        "SELECT \n"
        "    branch_id, branch_name, total_capacity, full_date, hour_of_day, academic_term_phase,\n"
        "    total_entries, total_exits, (total_entries - total_exits) AS net_flow,\n"
        "    SUM(total_entries - total_exits) OVER (\n"
        "        PARTITION BY branch_id, full_date ORDER BY hour_of_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n"
        "    ) AS estimated_current_occupancy,\n"
        "    ROUND(\n"
        "        (GREATEST(0, SUM(total_entries - total_exits) OVER (\n"
        "            PARTITION BY branch_id, full_date ORDER BY hour_of_day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n"
        "        ))::NUMERIC / total_capacity) * 100, 2\n"
        "    ) AS capacity_utilization_pct\n"
        "FROM HourlyFlow;\n\n"
        "CREATE OR REPLACE VIEW view_kpi_college_usage_share AS\n"
        "SELECT \n"
        "    p.college_affiliation, p.patron_type,\n"
        "    COUNT(DISTINCT p.patron_id) AS total_registered_patrons,\n"
        "    COUNT(DISTINCT t.turnstile_event_id) AS total_library_visits,\n"
        "    COUNT(DISTINCT c.circulation_id) AS total_physical_books_borrowed,\n"
        "    COUNT(DISTINCT d.access_event_id) AS total_digital_sessions,\n"
        "    COUNT(CASE WHEN d.access_action = 'PDF_DOWNLOAD' THEN 1 END) AS total_research_downloads\n"
        "FROM dim_patron p\n"
        "LEFT JOIN fact_physical_turnstile t ON p.patron_id = t.patron_id\n"
        "LEFT JOIN fact_physical_circulation c ON p.patron_id = c.patron_id\n"
        "LEFT JOIN fact_digital_access d ON p.patron_id = d.patron_id\n"
        "GROUP BY p.college_affiliation, p.patron_type;"
    )

    add_h2("Appendix C: Power BI DAX Measures Reference")
    add_body("Summary table of all DAX measures developed and deployed across the Power BI analytical layer:")

    # Table 8.1
    t81_data = [
        ["Measure Name", "DAX Expression", "Target Formatting", "Reporting Location"],
        ["Total Digital Spend", "SUM(view_kpi_publisher_roi[annual_subscription_cost])", "Currency ($#,##0)", "Page 1 & 2 Cards"],
        ["Total Downloads", "SUM(view_kpi_publisher_roi[total_pdf_downloads])", "Whole Number (#,##0)", "Page 1 & 2 Cards"],
        ["Overall CPD", "DIVIDE([Total Digital Spend], [Total Downloads], 0)", "Currency ($#,##0.00)", "Page 1 & 2 Cards"],
        ["Potential Annual Savings", "CALCULATE(SUM(...), Action IN {'CANCEL...','RENEGOTIATE...'})", "Currency ($#,##0)", "Page 1 & 2 Cards"],
        ["Peak Capacity %", "DIVIDE(MAX(view_kpi_hourly_occupancy[capacity_utilization_pct]), 100, 0)", "Percentage (0.0%)", "Page 3 Gauge Dial"]
    ]
    t81 = doc.add_table(rows=len(t81_data), cols=4)
    set_table_borders(t81)
    t81.alignment = WD_TABLE_ALIGNMENT.CENTER
    for r_idx, row in enumerate(t81_data):
        for c_idx, val in enumerate(row):
            cell = t81.cell(r_idx, c_idx)
            set_cell_margins(cell, top=100, bottom=100, left=150, right=150)
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            run = p.add_run(val)
            run.font.name = 'Times New Roman'
            run.font.size = Pt(11)
            run.font.color.rgb = BLACK
            if r_idx == 0:
                run.font.bold = True
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    add_body("Table 8.1: Complete Power BI DAX Measure Inventory and Formatting Specifications", italic=True)

    # Save document
    doc_filename = "UCB_Library_Analytics_Report.docx"
    doc.save(doc_filename)
    print(f"Report successfully compiled and saved as: {doc_filename}")

if __name__ == "__main__":
    create_report()